# 03 - Publish the adapted dataset to Hugging Face and Kaggle

**adaption-devkit** is a *community, unofficial* toolkit. **Not affiliated with or endorsed by Adaption Labs.** Apache-2.0. Author: Aivaras Navardauskas (MANIFESTA), GitHub A1VARA5.

This notebook downloads the adapted dataset and releases it to **Hugging Face** and **Kaggle** with a card and a cover image.

All outputs are **illustrative**; this notebook was authored, not executed.

## Why we publish manually (the 501 reason)

Adaption exposes a publish endpoint (`POST /api/v1/datasets/{id}/publish`), but it currently returns **`501 - not yet implemented`**. So for the hackathon's required Hugging Face + Kaggle releases we **do not rely on it**. Instead we:

1. `download` the processed dataset from Adaption (a presigned URL).
2. Push to Hugging Face with `huggingface_hub` and to Kaggle with the `kaggle` CLI.

If the devkit's optional `adaption_kit.publish` helper is installed, we use it as a thin wrapper; otherwise we fall back to the raw libraries. Re-check the 501 endpoint before final submission in case it ships.

## Setup and auth

Three separate credentials, all from the environment:

```bash
export ADAPTION_BASE_URL="https://api.prod.adaptionlabs.ai"
export ADAPTION_API_KEY="pt_live_..."
export HF_TOKEN="hf_..."                 # Hugging Face write token
# Kaggle: place kaggle.json at ~/.kaggle/kaggle.json, OR set:
export KAGGLE_USERNAME="your-handle"
export KAGGLE_KEY="..."
```

In [ ]:
%pip install -q adaption huggingface_hub kaggle pandas

In [ ]:
import os
from adaption import Adaption

BASE_URL = os.environ["ADAPTION_BASE_URL"]
assert os.environ.get("ADAPTION_API_KEY"), "Set ADAPTION_API_KEY first."
client = Adaption(base_url=BASE_URL)

# The dataset_id from notebook 01 or 02. Paste the real one here.
dataset_id = os.environ.get("ADAPTION_DATASET_ID", "ds_REPLACE_ME")
print("publishing dataset:", dataset_id)

## Step 1 - Download the adapted dataset

`download` returns a presigned URL. It works once a run has produced output (status `ready`, or `failed` for partial recovery). It returns `422` only if no run ever started. We pull it to a local file.

In [ ]:
import urllib.request

OUT_DIR = "published"
os.makedirs(OUT_DIR, exist_ok=True)
LOCAL_FILE = os.path.join(OUT_DIR, "adapted_dataset.csv")

url = client.datasets.download(dataset_id, file_format="csv")  # presigned S3 URL
urllib.request.urlretrieve(url, LOCAL_FILE)
print("downloaded to", LOCAL_FILE)

# Quick sanity check, reading with utf-8-sig to strip any BOM.
import pandas as pd
df = pd.read_csv(LOCAL_FILE, encoding="utf-8-sig")
print(f"{len(df)} rows, columns: {list(df.columns)}")  # illustrative

## Step 2 - Build a card and a cover image

A good release needs a README/card and a cover. If `adaption_kit` ships card helpers we use them; otherwise we write a minimal card inline. Either way the content is the same shape.

In [ ]:
HF_REPO_ID = "your-hf-username/brandvoice-adapted"      # change me
KAGGLE_SLUG = "your-kaggle-username/brandvoice-adapted"  # change me

# Try the optional devkit card helpers; fall back to an inline card if absent.
try:
    from adaption_kit import generate_dataset_card
    CARD_MD = generate_dataset_card(
        title="BrandVoice Adapted (community sample)",
        summary="Marketing copy corpus adapted with Adaption Adaptive Data.",
        license="apache-2.0",
        source="Adapted via Adaption Adaptive Data (community, unofficial devkit).",
    )
except Exception:
    CARD_MD = (
        "---\n"
        "license: apache-2.0\n"
        "tags: [marketing, synthetic, adaption-adaptive-data]\n"
        "---\n\n"
        "# BrandVoice Adapted (community sample)\n\n"
        "Marketing copy corpus adapted with Adaption Adaptive Data.\n\n"
        "Produced with the **community, unofficial** adaption-devkit. "
        "Not affiliated with or endorsed by Adaption Labs.\n"
    )

CARD_PATH = os.path.join(OUT_DIR, "README.md")
with open(CARD_PATH, "w", encoding="utf-8") as f:
    f.write(CARD_MD)
print("wrote card to", CARD_PATH)

In [ ]:
# A tiny generated cover so the release has a visual. Pillow only; no external assets.
COVER_PATH = os.path.join(OUT_DIR, "cover.png")
try:
    from PIL import Image, ImageDraw
    img = Image.new("RGB", (1200, 630), (18, 20, 28))
    d = ImageDraw.Draw(img)
    d.text((60, 280), "BrandVoice Adapted", fill=(235, 235, 245))
    d.text((60, 320), "adaption-devkit (community, unofficial)", fill=(150, 160, 180))
    img.save(COVER_PATH)
    print("wrote cover to", COVER_PATH)
except Exception as e:
    COVER_PATH = None
    print("cover skipped (Pillow not installed):", e)

## Step 3 - Publish

Preferred path: the optional `adaption_kit.publish` helper, which wraps both targets behind one call. If it is not present, we fall back to **raw `huggingface_hub`** and the **`kaggle` CLI** directly. Both paths authenticate purely from environment variables.

In [ ]:
published_via_kit = False
try:
    from adaption_kit import publish as kit_publish  # optional devkit helper
    kit_publish.to_huggingface(
        repo_id=HF_REPO_ID,
        files=[LOCAL_FILE, CARD_PATH] + ([COVER_PATH] if COVER_PATH else []),
        token=os.environ["HF_TOKEN"],
    )
    kit_publish.to_kaggle(
        slug=KAGGLE_SLUG,
        folder=OUT_DIR,
        title="BrandVoice Adapted (community sample)",
    )
    published_via_kit = True
    print("published via adaption_kit.publish")
except Exception as e:
    # Helper not installed, or it raised; fall back to the raw libraries below.
    print("adaption_kit.publish unavailable, using raw libraries:", e)

### Fallback A - Hugging Face via `huggingface_hub`

In [ ]:
if not published_via_kit:
    from huggingface_hub import HfApi

    api = HfApi(token=os.environ["HF_TOKEN"])  # token from env, never hardcoded
    api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)

    api.upload_file(path_or_fileobj=LOCAL_FILE, path_in_repo="adapted_dataset.csv",
                    repo_id=HF_REPO_ID, repo_type="dataset")
    api.upload_file(path_or_fileobj=CARD_PATH, path_in_repo="README.md",
                    repo_id=HF_REPO_ID, repo_type="dataset")
    if COVER_PATH:
        api.upload_file(path_or_fileobj=COVER_PATH, path_in_repo="cover.png",
                        repo_id=HF_REPO_ID, repo_type="dataset")
    print("pushed to https://huggingface.co/datasets/" + HF_REPO_ID)

### Fallback B - Kaggle via the `kaggle` CLI

Kaggle needs a `dataset-metadata.json` next to the data files. We write one, then call the CLI. The CLI reads credentials from `~/.kaggle/kaggle.json` or from `KAGGLE_USERNAME` / `KAGGLE_KEY`.

In [ ]:
if not published_via_kit:
    import json
    import subprocess

    owner, slug = KAGGLE_SLUG.split("/", 1)
    meta = {
        "title": "BrandVoice Adapted (community sample)",
        "id": KAGGLE_SLUG,
        "licenses": [{"name": "Apache 2.0"}],
    }
    with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    # First publish: create-new. For later versions use: dataset version -p OUT_DIR -m "msg"
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", OUT_DIR, "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    print("see https://www.kaggle.com/datasets/" + KAGGLE_SLUG)

## Recap

- The Adaption **publish endpoint is 501**, so we released manually.
- We `download`ed the adapted dataset, wrote a **card + cover**, and pushed to **Hugging Face** and **Kaggle**.
- We preferred `adaption_kit.publish` when present, and fell back to raw `huggingface_hub` + the `kaggle` CLI.
- Every credential came from an **environment variable**.

That completes the loop: ingest -> adapt -> evaluate -> export -> publish.